# 02 — Experimentos: comparación de modelos y escalabilidad

**Proyecto Big Data — NYC Taxi · Spark MLlib**

Comparación de 4 clasificadores distribuidos (Regresión Logística, Árbol, Random Forest, GBT) con ponderación de clases, y benchmarks de escalabilidad. La lógica reside en el paquete `nyc_taxi_mllib`.

In [ ]:
import sys; sys.path.insert(0, '../src')
from nyc_taxi_mllib import (build_spark, load_trips, clean_trips, add_label,
    add_class_weights, build_feature_pipeline, make_models, time_fit,
    evaluate_predictions, class_balance, data_scaling)
spark = build_spark('experiments-notebook')

## 1. Preparación de datos y variables
Limpieza → etiqueta → ponderación de clases → pipeline de features (`StringIndexer` + `OneHotEncoder` + `VectorAssembler`).

In [ ]:
prepared = add_class_weights(add_label(clean_trips(load_trips(spark)))).cache()
print(class_balance(prepared))
feat_model = build_feature_pipeline().fit(prepared)
featurized = feat_model.transform(prepared).select('features','label','class_weight').cache()
featurized.count()

## 2. Muestra y partición
Muestra de ~2M viajes, split 80/20 idéntico para todos los modelos.

In [ ]:
sample = featurized.sample(False, 2_000_000/prepared.count(), seed=42).cache()
train, test = sample.randomSplit([0.8, 0.2], seed=42)
train.cache().count(), test.cache().count()

## 3. Entrenamiento y evaluación de los 4 modelos
Métricas: AUC, F1, accuracy y **recall de la clase minoritaria** (la métrica honesta).

In [ ]:
results = {}
for name, est in make_models().items():
    secs, model = time_fit(est, train)
    m = evaluate_predictions(model.transform(test)); m['train_seconds'] = secs
    results[name] = m
    print(f"{name:18s} AUC={m['auc']:.3f} F1={m['f1']:.3f} "
          f"acc={m['accuracy']:.3f} recall+={m['recall_pos']:.3f} t={secs:.1f}s")

> **Hallazgo:** la *accuracy* engaña bajo desbalance (un trivial daría ~90.6%). GBT y Random Forest dan el mejor AUC; la elección depende del costo de cómputo.

## 4. Escalabilidad de datos
Tiempo de entrenamiento de la Regresión Logística vs. volumen.

In [ ]:
lr = make_models(weight_col=None)['LogisticRegression']
for r in data_scaling(lr, featurized, fractions=[0.05, 0.1, 0.25, 0.5, 1.0], seed=42):
    print(f"n={r['n_rows']:>10,}  t={r['train_seconds']:.1f}s")

La escalabilidad fuerte (vs. núcleos) está en `scripts/run_strong_scaling.py` (requiere recrear la sesión Spark con distinto nº de cores).

In [ ]:
spark.stop()